In [2]:
from pynq import Overlay
import time


In [15]:
# ==========================================
# 1. 載入硬體 Overlay
# ==========================================
print("載入硬體中...")
# 請將 "your_design.bit" 替換成你實際的 bit 檔名 (確保 hwh 檔也在同目錄)
overlay = Overlay("RFC8439_1.bit") 

# 綁定 GPIO 與 BRAM 控制器
gpio_len  = overlay.axi_gpio_0
gpio_ctrl = overlay.axi_gpio_1
bram_src  = overlay.axi_bram_ctrl_0
bram_dst  = overlay.axi_bram_ctrl_1

# ==========================================
# 2. 設定參數 (MSG_LEN = 4096, AD_LEN = 0)
# ==========================================
MSG_LEN = 16768 #以gen_RFC.py上面寫的長度為準，通常為明文長度
AD_LEN = 0 #nonce跟padding中間的長度

gpio_len.channel1.write(MSG_LEN, 0xFFFF_FFFF)
gpio_len.channel2.write(AD_LEN, 0xFFFF_FFFF)
print(f"✅ 參數設定完成: msg_length={MSG_LEN}, ad_length={AD_LEN}")

# ==========================================
# 3. 讀取 Src_RAM_data.txt 並寫入 BRAM0
# ==========================================
print("📥 正在將 Src_RAM_data.txt 寫入 BRAM0...")
with open("stress_Src_RAM_enc.txt", "r") as f:
    src_lines = f.readlines()

for i, line in enumerate(src_lines):
    # 移除換行符號並將 16 進位字串轉為整數
    data = int(line.strip(), 16) 
    # BRAM 是 byte-addressable，每個 32-bit word 佔 4 bytes，所以 offset 要乘 4
    bram_src.write(i * 4, data) 


print("✅ BRAM0 寫入完成！")
print("🔍 正在從 BRAM0 讀回前 10 筆資料進行肉眼檢查...")
for i in range(10):
    # 用 .read(offset) 讀取，offset 記得是 byte address (i * 4)
    val = bram_src.read(i * 4)
    
    # 格式化輸出：補齊 8 位數的 16 進位字串
    print(f"Word {i:02d} (Offset 0x{i*4:04x}): 0x{val:08x}")

載入硬體中...
✅ 參數設定完成: msg_length=16768, ad_length=0
📥 正在將 Src_RAM_data.txt 寫入 BRAM0...
✅ BRAM0 寫入完成！
🔍 正在從 BRAM0 讀回前 10 筆資料進行肉眼檢查...
Word 00 (Offset 0x0000): 0x7958bc1a
Word 01 (Offset 0x0004): 0x10742563
Word 02 (Offset 0x0008): 0x58adbcef
Word 03 (Offset 0x000c): 0x416a3975
Word 04 (Offset 0x0010): 0x89254137
Word 05 (Offset 0x0014): 0x54022163
Word 06 (Offset 0x0018): 0xebad9678
Word 07 (Offset 0x001c): 0x80adeccf
Word 08 (Offset 0x0020): 0x7925de1a
Word 09 (Offset 0x0024): 0xdabc8263


In [17]:

# ==========================================
# 4. 觸發 start 訊號
# ==========================================
# gpio_ctrl.channel1 (輸出): bit 0 = start, bit 1 = mode_decrypt
# 0x1 (二進位 01) 代表啟動硬體，且 mode_decrypt = 0 (假設為加密)
print("🚀 啟動 RFC8439 IP...")
print("--- 記憶體抽查 ---")
print(f"[偏移量 0] 預期是 Key的第一個Word: {hex(bram_src.read(0))}")
print(f"[偏移量 {48+AD_LEN}] 預期是 明文的第一個Word: {hex(bram_src.read(48+AD_LEN))}")
print("------------------")
gpio_ctrl.channel1.write(0x0, 0x3) 
time.sleep(0.01) # 給硬體一點反應時間回到 IDLE

print("🚀 啟動 RFC8439 IP...")
gpio_ctrl.channel1.write(0x1, 0x3)

# ==========================================
# 5. 等待 done 訊號回傳
# ==========================================
print("⏳ 等待硬體運算...")
start_time = time.time()
timeout = 5.0 # 設定 5 秒 Timeout 防當機

while True:
    # 讀取 gpio_ctrl.channel2 (輸入): bit 0 = done, bit 1 = mac_error
    status = gpio_ctrl.channel2.read()
    
    # 檢查 bit 0 是否為 1
    if (status & 0x1) != 0:
        break
        
    # Timeout 檢查
    if (time.time() - start_time) > timeout:
        print("❌ 錯誤：硬體逾時 (Timeout)！請檢查 IP 狀態機是否卡死。")
        break

# 檢查是否運算成功
if (status & 0x1) != 0:
    print("✅ 硬體運算完成！")
    # 順便檢查 bit 1 (mac_error)
    if (status & 0x2) != 0:
        print("⚠️ 警告：偵測到 MAC Error 訊號！")
        
# 【重要】將 start 訊號拉低，確保下次能產生 rising edge
gpio_ctrl.channel1.write(0x0,0x3)

# ==========================================
# 6. 比對 BRAM1 與 Dst_RAM_data.txt
# ==========================================
print("📤 正在從 BRAM1 讀取資料並比對 Dst_RAM_data.txt...")
with open("stress_Dst_RAM_enc.txt", "r") as f:
    dst_lines = f.readlines()

error_count = 0
for i, line in enumerate(dst_lines):
    expected_data = int(line.strip(), 16)
    
    # 從 BRAM1 讀取運算結果
    actual_data = bram_dst.read(i * 4)
    
    if actual_data != expected_data:
        # 為了避免畫面洗版，只印出前 10 個錯誤
        if error_count < 10:
            print(f"❌ 錯誤 @ Word {i} (Offset 0x{i*4:04x}): 預期 0x{expected_data:08x}, 實際出 0x{actual_data:08x}")
        error_count += 1

if error_count == 0:
    print("🎉 恭喜！資料比對完全正確，硬體驗證成功！")
else:
    print(f"💔 驗證失敗：總共有 {error_count} 個 Word 與預期不符。")
    
gpio_counter = overlay.axi_gpio_2
hw_cycles = gpio_counter.channel1.read()

# 4. 奈秒級轉換 (100MHz = 10ns per cycle)
CLOCK_PERIOD_NS = 25
hardware_time_ns = hw_cycles * CLOCK_PERIOD_NS
hardware_time_us = hardware_time_ns / 1000

print(f"⏱️ 硬體消耗週期: {hw_cycles} cycles")
print(f"🚀 精準硬體處理時間: {hardware_time_ns} 奈秒 ({hardware_time_us:.2f} 微秒)")

# 5. 計算真實吞吐量 (Throughput)
throughput_MBps = (MSG_LEN / 1024 / 1024) / (hardware_time_ns * 1e-9)
print(f"⚡ 硬體吞吐量: {throughput_MBps:.2f} MB/s")

🚀 啟動 RFC8439 IP...
--- 記憶體抽查 ---
[偏移量 0] 預期是 Key的第一個Word: 0x7958bc1a
[偏移量 48] 預期是 明文的第一個Word: 0x12345678
------------------
🚀 啟動 RFC8439 IP...
⏳ 等待硬體運算...
✅ 硬體運算完成！
📤 正在從 BRAM1 讀取資料並比對 Dst_RAM_data.txt...
🎉 恭喜！資料比對完全正確，硬體驗證成功！
⏱️ 硬體消耗週期: 21058 cycles
🚀 精準硬體處理時間: 526450 奈秒 (526.45 微秒)
⚡ 硬體吞吐量: 30.38 MB/s
